In [1]:
from typing import Optional, Dict, List, Any
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field, field_validator
import nest_asyncio
import uvicorn
import threading
import subprocess
import requests
import re
import time


ASSESSMENT_WEIGHT = 0.40
SIMULATION_WEIGHT = 0.60

DEFAULT_EXPECTED_RESPONSE_TIME = 15.0

print("========================================")
print("AI Configuration")
print("========================================")
print("Dynamic Indicators: ENABLED")
print("Dynamic Careers: ENABLED")
print("Dynamic Weights: ENABLED")
print("========================================")

AI Configuration
Dynamic Indicators: ENABLED
Dynamic Careers: ENABLED
Dynamic Weights: ENABLED


In [2]:
class IndicatorDefinition(BaseModel):

    indicator_id: str

    name: str

    # arabic display name
    name_ar: Optional[str] = None

    # how this indicator is calculated
    calculation_type: str = "manual"

    # optional configuration for calculation
    config: Dict[str, Any] = Field(
        default_factory=dict
    )


class BehaviorEvent(BaseModel):

    responseSeconds: float = Field(
        default=15,
        ge=0
    )

    attempts: int = Field(
        default=1,
        ge=1
    )

    correct: Optional[bool] = None

    changedDecision: bool = False

    usedHint: bool = False

    creativeAction: bool = False

    requestedTeamHelp: bool = False

    explainedDecision: bool = False

    leadershipAction: bool = False

    expectedResponseTime: Optional[float] = Field(
        default=None,
        ge=0
    )

    simulation_id: Optional[str] = None

    challenge_type: Optional[str] = None


    metadata: Dict[str, Any] = Field(
        default_factory=dict
    )


class AssessmentScores(BaseModel):


    scores: Dict[str, float] = Field(
        default_factory=dict
    )

    @field_validator("scores")
    @classmethod
    def validate_scores(cls, value):

        for indicator_id, score in value.items():

            if score < 0 or score > 100:

                raise ValueError(
                    f"Score for '{indicator_id}' "
                    f"must be between 0 and 100."
                )

        return value

In [3]:

class CareerProfile(BaseModel):

    career_id: str

    career_name: str

    weights: Dict[str, float]

    @field_validator("weights")
    @classmethod
    def validate_weights(cls, value):

        if not value:

            raise ValueError(
                "Career must have at least one indicator weight."
            )

        for indicator_id, weight in value.items():

            if weight < 0 or weight > 1:

                raise ValueError(
                    f"Weight for '{indicator_id}' "
                    f"must be between 0 and 1."
                )

        total = sum(value.values())

        if abs(total - 1.0) > 0.001:

            raise ValueError(
                f"Career weights must equal 1.0 (100%). "
                f"Received: {round(total, 4)}"
            )

        return value

In [4]:
class AnalysisRequest(BaseModel):

    student_id: str

    indicators: List[IndicatorDefinition] = Field(
        default_factory=list
    )


    assessment_scores: Optional[AssessmentScores] = None


    behavior_events: List[BehaviorEvent] = Field(
        default_factory=list
    )


    career_profiles: List[CareerProfile] = Field(
        default_factory=list
    )

In [5]:
def validate_indicator_configuration(
    indicators,
    career_profiles,
    assessment_scores
):

    indicator_ids = {
        indicator.indicator_id
        for indicator in indicators
    }


    for career in career_profiles:

        unknown_indicators = (
            set(career.weights.keys())
            - indicator_ids
        )

        if unknown_indicators:

            raise ValueError(
                f"Career '{career.career_name}' "
                f"uses unknown indicators: "
                f"{', '.join(unknown_indicators)}"
            )


    if assessment_scores:

        unknown_assessment_indicators = (
            set(assessment_scores.scores.keys())
            - indicator_ids
        )

        if unknown_assessment_indicators:

            raise ValueError(
                "Assessment contains unknown indicators: "
                +
                ", ".join(
                    unknown_assessment_indicators
                )
            )

    return True

In [6]:
def calculate_indicator_score(
    indicator,
    events
):

    calculation_type = (
        indicator.calculation_type
    )

    config = indicator.config


    if not events:

        return None

    if calculation_type == "manual":

        return None


    if calculation_type == "accuracy":

        valid_events = [
            event
            for event in events
            if event.correct is not None
        ]

        if not valid_events:
            return None

        correct_count = sum(
            1
            for event in valid_events
            if event.correct
        )

        return round(
            (
                correct_count
                / len(valid_events)
            ) * 100,
            2
        )


    if calculation_type == "persistence":

        scores = []

        for event in events:

            if (
                event.correct is True
                and event.attempts > 1
            ):

                score = 95

            elif event.correct is True:

                score = 80

            elif event.attempts >= 3:

                score = 75

            elif event.attempts == 2:

                score = 60

            else:

                score = 40

            scores.append(score)

        return round(
            sum(scores) / len(scores),
            2
        )


    if calculation_type == "response_time":

        scores = []

        target_time = config.get(
            "expectedResponseTime",
            DEFAULT_EXPECTED_RESPONSE_TIME
        )

        for event in events:

            expected_time = (
                event.expectedResponseTime
                if event.expectedResponseTime is not None
                else target_time
            )

            difference = abs(
                event.responseSeconds
                - expected_time
            )

            score = 100 - (
                difference * 4
            )

            scores.append(
                max(
                    0,
                    min(100, score)
                )
            )

        return round(
            sum(scores) / len(scores),
            2
        )


    if calculation_type == "creative_action":

        creative_count = sum(
            1
            for event in events
            if event.creativeAction
        )

        ratio = (
            creative_count
            / len(events)
        )

        return round(
            40 + (60 * ratio),
            2
        )


    if calculation_type == "communication":

        relevant = [
            event
            for event in events
            if (
                event.requestedTeamHelp
                or event.explainedDecision
            )
        ]

        if not relevant:
            return None

        scores = []

        for event in relevant:

            score = 50

            if event.requestedTeamHelp:
                score += 20

            if event.explainedDecision:
                score += 30

            scores.append(
                min(100, score)
            )

        return round(
            sum(scores) / len(scores),
            2
        )


    if calculation_type == "leadership":

        relevant = [
            event
            for event in events
            if event.leadershipAction
        ]

        if not relevant:
            return None

        return 70.0


    return None

In [7]:
def extract_behavioral_features(
    indicators,
    events
):

    behavioral_profile = {}

    for indicator in indicators:

        score = calculate_indicator_score(
            indicator,
            events
        )

        behavioral_profile[
            indicator.indicator_id
        ] = score

    return behavioral_profile

In [8]:
def combine_scores(
    indicators,
    assessment_scores,
    simulation_scores
):

    final_scores = {}

    assessment_dict = (
        assessment_scores.scores
        if assessment_scores
        else {}
    )

    for indicator in indicators:

        indicator_id = (
            indicator.indicator_id
        )

        assessment_value = (
            assessment_dict.get(
                indicator_id
            )
        )

        simulation_value = (
            simulation_scores.get(
                indicator_id
            )
        )


        if (
            assessment_value is not None
            and simulation_value is not None
        ):

            final_scores[indicator_id] = round(
                (
                    assessment_value
                    * ASSESSMENT_WEIGHT
                )
                +
                (
                    simulation_value
                    * SIMULATION_WEIGHT
                ),
                2
            )


        elif assessment_value is not None:

            final_scores[indicator_id] = round(
                assessment_value,
                2
            )


        elif simulation_value is not None:

            final_scores[indicator_id] = round(
                simulation_value,
                2
            )

        else:

            final_scores[indicator_id] = None

    return final_scores

In [9]:
def calculate_career_score(
    behavioral_profile,
    weights
):

    weighted_score = 0.0
    available_weight = 0.0

    total_weight = sum(
        weights.values()
    )

    if total_weight <= 0:

        return None, 0.0

    for indicator_id, weight in weights.items():

        score = behavioral_profile.get(
            indicator_id
        )

        if score is None:
            continue

        weighted_score += (
            score * weight
        )

        available_weight += weight

    if available_weight <= 0:

        return None, 0.0

    score = (
        weighted_score
        / available_weight
    )

    data_coverage = (
        available_weight
        / total_weight
    )

    return (
        round(
            max(0, min(100, score)),
            2
        ),

        round(
            data_coverage,
            2
        )
    )


def get_indicator_name(
    indicator_id,
    indicators
):

    for indicator in indicators:

        if indicator.indicator_id == indicator_id:

            return (
                indicator.name_ar
                or indicator.name
            )

    return indicator_id


def get_career_strengths(
    behavioral_profile,
    weights,
    indicators
):

    contributions = []

    for indicator_id, weight in weights.items():

        score = behavioral_profile.get(
            indicator_id
        )

        if score is None:
            continue

        contribution = (
            score * weight
        )

        contributions.append({

            "indicator":
                indicator_id,

            "name":
                get_indicator_name(
                    indicator_id,
                    indicators
                ),

            "score":
                round(score, 2),

            "weight":
                round(weight, 2),

            "contribution":
                round(contribution, 2)
        })

    contributions.sort(
        key=lambda x: x["contribution"],
        reverse=True
    )

    return contributions[:3]


def get_development_areas(
    behavioral_profile,
    weights,
    indicators
):

    areas = []

    for indicator_id, weight in weights.items():

        score = behavioral_profile.get(
            indicator_id
        )

        if score is None:
            continue

        if score < 70:

            areas.append({

                "indicator":
                    indicator_id,

                "name":
                    get_indicator_name(
                        indicator_id,
                        indicators
                    ),

                "score":
                    round(score, 2),

                "weight":
                    round(weight, 2),

                "gap":
                    round(
                        70 - score,
                        2
                    )
            })

    areas.sort(
        key=lambda x: x["weight"],
        reverse=True
    )

    return areas[:3]


def generate_recommendations(
    behavioral_profile: Dict[str, Optional[float]],
    career_profiles: List[CareerProfile],
    indicators: List[IndicatorDefinition],
    assessment_scores: Dict[str, float],
    simulation_scores: Dict[str, Optional[float]],
    behavior_events: List[BehaviorEvent]
):

    recommendations = []

    for career in career_profiles:

        score, _coverage = calculate_career_score(
            behavioral_profile,
            career.weights
        )


        career_evidence = calculate_career_evidence(
            career,
            indicators,
            assessment_scores,
            simulation_scores,
            behavior_events
        )


        strengths = get_career_strengths(
            behavioral_profile,
            career.weights,
            indicators
        )


        development_areas = get_development_areas(
            behavioral_profile,
            career.weights,
            indicators
        )

        recommendations.append({
            "career_id": career.career_id,
            "career_name": career.career_name,

            # Keep the actual match score separate
            "score": round(score, 2) if score is not None else None,

            # Evidence
            "dataCoverage": career_evidence["dataCoverage"],
            "evidenceQuality": career_evidence["evidenceQuality"],

            # Career matching factors
            "strengths": strengths,

            "developmentAreas": development_areas
        })

    # Sort by match score
    recommendations.sort(
        key=lambda item: item["score"],
        reverse=True
    )

    return recommendations

In [10]:
MIN_SIMULATIONS_FOR_HIGH_CONFIDENCE = 5
MIN_EVENTS_FOR_HIGH_CONFIDENCE = 30
MIN_CHALLENGE_TYPES_FOR_HIGH_CONFIDENCE = 3

HIGH_COVERAGE_THRESHOLD = 0.80
MEDIUM_COVERAGE_THRESHOLD = 0.50

CONSISTENCY_HIGH = 0.80
CONSISTENCY_MEDIUM = 0.60

# Weight of each evidence factor
EVIDENCE_WEIGHTS = {
    "simulationCount": 0.15,
    "challengeDiversity": 0.10,
    "behaviorEvents": 0.15,
    "assessmentAvailability": 0.10,
    "assessmentSimulationConsistency": 0.20,
    "dataCoverage": 0.15,
    "indicatorEvidenceQuality": 0.15,
}

In [11]:

def calculate_simulation_count(events: List[BehaviorEvent]) -> int:
    simulation_ids = {
        event.simulation_id
        for event in events
        if event.simulation_id
    }

    return len(simulation_ids)


def calculate_challenge_diversity(events: List[BehaviorEvent]) -> int:
    challenge_types = {
        event.challenge_type
        for event in events
        if event.challenge_type
    }

    return len(challenge_types)


def calculate_event_evidence(events: List[BehaviorEvent]) -> float:
    count = len(events)

    if count <= 0:
        return 0.0

    return min(
        count / MIN_EVENTS_FOR_HIGH_CONFIDENCE,
        1.0
    )


def calculate_simulation_evidence(events: List[BehaviorEvent]) -> float:
    count = calculate_simulation_count(events)

    if count <= 0:
        return 0.0

    return min(
        count / MIN_SIMULATIONS_FOR_HIGH_CONFIDENCE,
        1.0
    )


def calculate_challenge_diversity_score(
    events: List[BehaviorEvent]
) -> float:

    diversity = calculate_challenge_diversity(events)

    if diversity <= 0:
        return 0.0

    return min(
        diversity / MIN_CHALLENGE_TYPES_FOR_HIGH_CONFIDENCE,
        1.0
    )

In [12]:

def calculate_assessment_simulation_consistency(
    assessment_scores: Dict[str, float],
    simulation_scores: Dict[str, Optional[float]]
) -> Optional[float]:

    common_indicators = [
        indicator_id
        for indicator_id in assessment_scores
        if indicator_id in simulation_scores
        and simulation_scores[indicator_id] is not None
    ]

    if not common_indicators:
        return None

    differences = []

    for indicator_id in common_indicators:

        assessment_value = assessment_scores[indicator_id]
        simulation_value = simulation_scores[indicator_id]

        difference = abs(
            assessment_value - simulation_value
        )

        normalized_difference = difference / 100.0

        differences.append(normalized_difference)

    average_difference = sum(differences) / len(differences)

    consistency = 1.0 - average_difference

    return round(
        max(0.0, min(1.0, consistency)),
        4
    )

In [13]:

def calculate_indicator_evidence_quality(
    indicator_id: str,
    assessment_scores: Dict[str, float],
    simulation_scores: Dict[str, Optional[float]],
    events: List[BehaviorEvent]
) -> float:

    assessment_available = (
        indicator_id in assessment_scores
    )

    simulation_available = (
        indicator_id in simulation_scores
        and simulation_scores[indicator_id] is not None
    )

    evidence_sources = 0

    if assessment_available:
        evidence_sources += 1

    if simulation_available:
        evidence_sources += 1

    if evidence_sources == 0:
        return 0.0

    # Two independent sources = stronger evidence
    if evidence_sources == 2:
        base_quality = 1.0
    else:
        base_quality = 0.60

    event_quality = calculate_event_evidence(events)

    if simulation_available:
        base_quality = (
            base_quality * 0.70
            + event_quality * 0.30
        )

    return round(
        min(1.0, base_quality),
        4
    )

In [14]:

def calculate_evidence_quality(
    indicators: List[IndicatorDefinition],
    assessment_scores: Dict[str, float],
    simulation_scores: Dict[str, Optional[float]],
    behavior_events: List[BehaviorEvent]
) -> Dict[str, Any]:

    simulation_count = calculate_simulation_count(
        behavior_events
    )

    challenge_diversity = calculate_challenge_diversity(
        behavior_events
    )

    simulation_evidence = calculate_simulation_evidence(
        behavior_events
    )

    challenge_evidence = calculate_challenge_diversity_score(
        behavior_events
    )

    event_evidence = calculate_event_evidence(
        behavior_events
    )

    assessment_available = len(assessment_scores) > 0

    assessment_score = (
        1.0
        if assessment_available
        else 0.0
    )

    consistency = calculate_assessment_simulation_consistency(
        assessment_scores,
        simulation_scores
    )

    consistency_score = (
        consistency
        if consistency is not None
        else 0.0
    )


    total_indicators = len(indicators)

    available_indicators = 0

    for indicator in indicators:

        indicator_id = indicator.indicator_id

        has_assessment = (
            indicator_id in assessment_scores
        )

        has_simulation = (
            indicator_id in simulation_scores
            and simulation_scores[indicator_id] is not None
        )

        if has_assessment or has_simulation:
            available_indicators += 1

    data_coverage = (
        available_indicators / total_indicators
        if total_indicators > 0
        else 0.0
    )


    indicator_qualities = []

    for indicator in indicators:

        quality = calculate_indicator_evidence_quality(
            indicator.indicator_id,
            assessment_scores,
            simulation_scores,
            behavior_events
        )

        indicator_qualities.append(quality)

    indicator_evidence_quality = (
        sum(indicator_qualities)
        / len(indicator_qualities)
        if indicator_qualities
        else 0.0
    )


    evidence_score = (
        simulation_evidence
        * EVIDENCE_WEIGHTS["simulationCount"]
        +
        challenge_evidence
        * EVIDENCE_WEIGHTS["challengeDiversity"]
        +
        event_evidence
        * EVIDENCE_WEIGHTS["behaviorEvents"]
        +
        assessment_score
        * EVIDENCE_WEIGHTS["assessmentAvailability"]
        +
        consistency_score
        * EVIDENCE_WEIGHTS["assessmentSimulationConsistency"]
        +
        data_coverage
        * EVIDENCE_WEIGHTS["dataCoverage"]
        +
        indicator_evidence_quality
        * EVIDENCE_WEIGHTS["indicatorEvidenceQuality"]
    )

    return {
        "simulationCount": simulation_count,
        "challengeDiversity": challenge_diversity,
        "behaviorEvents": len(behavior_events),
        "assessmentAvailable": assessment_available,
        "assessmentSimulationConsistency": (
            round(consistency, 4)
            if consistency is not None
            else None
        ),
        "dataCoverage": round(data_coverage, 4),
        "indicatorEvidenceQuality": round(
            indicator_evidence_quality,
            4
        ),
        "evidenceScore": round(
            min(1.0, evidence_score),
            4
        )
    }

In [15]:

def calculate_career_evidence(
    career: CareerProfile,
    indicators: List[IndicatorDefinition],
    assessment_scores: Dict[str, float],
    simulation_scores: Dict[str, Optional[float]],
    behavior_events: List[BehaviorEvent]
) -> Dict[str, Any]:

    relevant_indicators = [
        indicator_id
        for indicator_id, weight in career.weights.items()
        if weight > 0
    ]

    available = 0
    qualities = []

    for indicator_id in relevant_indicators:

        has_assessment = (
            indicator_id in assessment_scores
        )

        has_simulation = (
            indicator_id in simulation_scores
            and simulation_scores[indicator_id] is not None
        )

        if has_assessment or has_simulation:
            available += 1

        quality = calculate_indicator_evidence_quality(
            indicator_id,
            assessment_scores,
            simulation_scores,
            behavior_events
        )

        qualities.append(
            quality
        )

    coverage = (
        available / len(relevant_indicators)
        if relevant_indicators
        else 0.0
    )

    evidence_quality = (
        sum(qualities) / len(qualities)
        if qualities
        else 0.0
    )

    return {
        "dataCoverage": round(
            coverage,
            4
        ),
        "evidenceQuality": round(
            evidence_quality,
            4
        )
    }

In [16]:

def get_top_strengths(
    behavioral_profile,
    indicators
):

    strengths = []

    for indicator_id, score in (
        behavioral_profile.items()
    ):

        if score is None:
            continue

        strengths.append({

            "indicator":
                indicator_id,

            "name":
                get_indicator_name(
                    indicator_id,
                    indicators
                ),

            "score":
                round(score, 2)
        })

    strengths.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return strengths[:5]

In [17]:

def calculate_confidence(
    evidence_score: float
) -> float:


    return round(
        max(0.0, min(0.95, evidence_score)),
        2
    )

In [18]:

app = FastAPI(
    title="Istilham AI Service",
    version="3.0.0"
)

app.add_middleware(
    CORSMiddleware,

    allow_origins=["*"],

    allow_credentials=True,

    allow_methods=["*"],

    allow_headers=["*"]
)


@app.get("/")
def root():

    return {

        "success": True,

        "service":
            "Istilham AI Service",

        "version":
            "3.0.0",

        "status":
            "running",

        "dynamic_indicators":
            True,

        "dynamic_careers":
            True,

        "dynamic_weights":
            True
    }


@app.get("/health")
def health():

    return {

        "success": True,

        "status":
            "healthy",

        "version":
            "3.0.0",

        "dynamic_indicators":
            True,

        "dynamic_careers":
            True,

        "dynamic_weights":
            True
    }


@app.post("/analyze")
def analyze(request: AnalysisRequest):

    try:


        if not request.indicators:

            raise HTTPException(
                status_code=400,
                detail=(
                    "No indicators were provided. "
                    "Backend must send the current "
                    "indicator definitions."
                )
            )


        if not request.career_profiles:

            raise HTTPException(
                status_code=400,
                detail=(
                    "No career profiles were provided."
                )
            )


        validate_indicator_configuration(
            request.indicators,
            request.career_profiles,
            request.assessment_scores
        )


        simulation_scores = (
            extract_behavioral_features(
                request.indicators,
                request.behavior_events
            )
        )

        assessment_scores_dict = (
            request.assessment_scores.scores
            if request.assessment_scores
            else {}
        )

        evidence = calculate_evidence_quality(
            indicators=request.indicators,
            assessment_scores=assessment_scores_dict,
            simulation_scores=simulation_scores,
            behavior_events=request.behavior_events
        )

        confidence = calculate_confidence(
            evidence["evidenceScore"]
        )


        behavioral_profile = combine_scores(
            request.indicators,
            request.assessment_scores,
            simulation_scores
        )


        strengths = get_top_strengths(
            behavioral_profile,
            request.indicators
        )


        recommendations = generate_recommendations(
            behavioral_profile=behavioral_profile,
            career_profiles=request.career_profiles,
            indicators=request.indicators,
            assessment_scores=assessment_scores_dict,
            simulation_scores=simulation_scores,
            behavior_events=request.behavior_events
        )


        available_indicators = sum(
            1
            for score in behavioral_profile.values()
            if score is not None
        )

        total_indicators = len(
            request.indicators
        )

        indicator_coverage = round(
            available_indicators
            / total_indicators,
            2
        )


        return {

            "success": True,

            "data": {

                "student_id":
                    request.student_id,

                "behavioral_profile":
                    behavioral_profile,

                "simulation_analysis":
                    simulation_scores,

                "strengths":
                    strengths,

                "recommendations":
                    recommendations,

                "confidence":
                    confidence,

                "data_quality": {

                    "availableIndicators":
                        available_indicators,

                    "totalIndicators":
                        total_indicators,

                    "indicatorCoverage":
                        indicator_coverage,

                    "behaviorEvents":
                        len(
                            request.behavior_events
                        ),

                    "careersAnalyzed":
                        len(
                            request.career_profiles
                        )
                }
            }
        }

    except HTTPException:
        raise

    except Exception as e:

        raise HTTPException(
            status_code=500,
            detail=f"Analysis failed: {str(e)}"
        )

In [19]:

nest_asyncio.apply()


def run_server():

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )


server_thread = threading.Thread(
    target=run_server,
    daemon=True
)

server_thread.start()

time.sleep(3)

print("========================================")
print("FastAPI started successfully")
print("========================================")
print("Dynamic Indicators : ENABLED")
print("Dynamic Careers    : ENABLED")
print("Dynamic Weights    : ENABLED")
print("Port               : 8000")
print("========================================")

INFO:     Started server process [635]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


FastAPI started successfully
Dynamic Indicators : ENABLED
Dynamic Careers    : ENABLED
Dynamic Weights    : ENABLED
Port               : 8000


In [20]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

cloudflare_process = subprocess.Popen(
    [
        "/content/cloudflared",
        "tunnel",
        "--url",
        "http://localhost:8000",
        "--no-autoupdate"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

cloudflare_url = None

start_time = time.time()

while time.time() - start_time < 30:

    line = cloudflare_process.stdout.readline()

    if not line:
        continue

    print(line.strip())

    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        line
    )

    if match:

        cloudflare_url = match.group(0)

        break


print()
print("========================================")
print("AI SERVICE URL")
print("========================================")
print(cloudflare_url)
print("========================================")

2026-09-30T14:43:58Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-30T14:43:58Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-30T14:44:02Z INF +--------------------------------------------------------------------------------------------+
2026-09-30T14:44:02Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-30T14:44:02Z INF |  https://action-all-kingston-eleven.trycloudflare.com 

In [21]:
# import requests
# import json

# cloudflare_url = cloudflare_url.rstrip("/")

# print("Testing AI Service:")
# print(cloudflare_url)


# indicators = [
#     {
#         "indicator_id": "problemSolving",
#         "name": "Problem Solving",
#         "name_ar": "حل المشكلات",
#         "calculation_type": "accuracy",
#         "config": {}
#     },
#     {
#         "indicator_id": "criticalThinking",
#         "name": "Critical Thinking",
#         "name_ar": "التفكير النقدي",
#         "calculation_type": "accuracy",
#         "config": {}
#     },
#     {
#         "indicator_id": "persistence",
#         "name": "Persistence",
#         "name_ar": "المثابرة",
#         "calculation_type": "persistence",
#         "config": {}
#     },
#     {
#         "indicator_id": "creativity",
#         "name": "Creativity",
#         "name_ar": "الإبداع",
#         "calculation_type": "creative_action",
#         "config": {}
#     },
#     {
#         "indicator_id": "decisionMaking",
#         "name": "Decision Making",
#         "name_ar": "اتخاذ القرار",
#         "calculation_type": "accuracy",
#         "config": {}
#     }
# ]


# assessment_scores = {
#     "problemSolving": 85,
#     "criticalThinking": 78,
#     "persistence": 92,
#     "creativity": 70,
#     "decisionMaking": 75
# }


# behavior_events = [
#     {
#         "responseSeconds": 12,
#         "attempts": 1,
#         "correct": True,
#         "changedDecision": False,
#         "usedHint": False,
#         "creativeAction": True,
#         "requestedTeamHelp": False,
#         "explainedDecision": True,
#         "leadershipAction": False,
#         "metadata": {}
#     },
#     {
#         "responseSeconds": 18,
#         "attempts": 2,
#         "correct": True,
#         "changedDecision": True,
#         "usedHint": False,
#         "creativeAction": False,
#         "requestedTeamHelp": True,
#         "explainedDecision": True,
#         "leadershipAction": False,
#         "metadata": {}
#     },
#     {
#         "responseSeconds": 14,
#         "attempts": 3,
#         "correct": True,
#         "changedDecision": False,
#         "usedHint": False,
#         "creativeAction": True,
#         "requestedTeamHelp": False,
#         "explainedDecision": True,
#         "leadershipAction": True,
#         "metadata": {}
#     }
# ]


# career_profiles = [
#     {
#         "career_id": "career_software",
#         "career_name": "Software Engineering",
#         "weights": {
#             "problemSolving": 0.30,
#             "criticalThinking": 0.30,
#             "persistence": 0.20,
#             "creativity": 0.10,
#             "decisionMaking": 0.10
#         }
#     },
#     {
#         "career_id": "career_cyber",
#         "career_name": "Cyber Security",
#         "weights": {
#             "problemSolving": 0.25,
#             "criticalThinking": 0.35,
#             "persistence": 0.20,
#             "creativity": 0.05,
#             "decisionMaking": 0.15
#         }
#     },
#     {
#         "career_id": "career_marketing",
#         "career_name": "Marketing",
#         "weights": {
#             "problemSolving": 0.10,
#             "criticalThinking": 0.15,
#             "persistence": 0.15,
#             "creativity": 0.40,
#             "decisionMaking": 0.20
#         }
#     }
# ]

# payload = {
#     "student_id": "dynamic_test_001",
#     "indicators": indicators,
#     "assessment_scores": {"scores": assessment_scores},
#     "behavior_events": behavior_events,
#     "career_profiles": career_profiles
# }


# try:
#     health_response = requests.get(
#         f"{cloudflare_url}/health",
#         timeout=30
#     )

#     print("\nHealth Check:")
#     print("Status:", health_response.status_code)
#     print("Response:", health_response.text)

# except Exception as e:
#     print("\nHealth Check Failed:")
#     print(repr(e))


# try:
#     response = requests.post(
#         f"{cloudflare_url}/analyze",
#         json=payload,
#         headers={
#             "Content-Type": "application/json"
#         },
#         timeout=120
#     )

#     print("\n" + "=" * 60)
#     print("ANALYZE RESULT")
#     print("=" * 60)

#     print("HTTP Status:", response.status_code)

#     try:
#         result = response.json()

#         print(
#             json.dumps(
#                 result,
#                 indent=2,
#                 ensure_ascii=False
#             )
#         )

#     except Exception:
#         print("Raw Response:")
#         print(response.text)

# except requests.exceptions.Timeout:
#     print("\nERROR: Request timed out.")

# except requests.exceptions.ConnectionError as e:
#     print("\nERROR: Connection failed.")
#     print(repr(e))

# except Exception as e:
#     print("\nERROR:")
#     print(repr(e))